
# Catchment Health Metrics (CHM) — Quickstart Notebook

This notebook demonstrates how to run the **CHM** package end‑to‑end using the example inputs under `tests/Input data/` and your own workspace directory.

**How to use this notebook**
1. **Edit the paths** in the next cell to match your environment (or keep the defaults to auto‑discover from `tests/Input data/`).  
2. Run each section step‑by‑step. Heavy/online steps (e.g., data downloads) are guarded by a switch so you can skip them during quick tests.
3. Each section begins with a brief explanation of what the function does and what you should expect as outputs.



## 1) Imports

We import CHM modules as a *functions-only* package. If installation succeeded (`pip install .` or `pip install -e .`), these imports should work.


In [ ]:

# --- Import with short aliases (functions-only package) ---
# NOTE: If these imports fail, ensure you've installed the package in this environment:
#   pip install -e .[dev]
try:
    import importlib.metadata as _im
    __chm_version = _im.version("chm")
except Exception:
    __chm_version = "unknown"

print(f"CHM package version: {__chm_version}")

from chm import dem_and_terrain
from chm import veg_indices_and_c_factor
from chm import surface_ground_water_connectivity
from chm import rusle_and_sdr_rusle
from chm import awap_historical_data
from chm import awra_historical_data
from chm import awra_projections_data
from chm import historical_bushfire_risk_profile
from chm import national_roads_risk_profile
from chm import landuse_risk_profile
from chm import appending_monitoring_data



## 2) Paths & configuration

- `CHM_Work_Space` — where outputs will be written (safe to create).  
- `Catchment_Shapefile_Path` — polygon boundary of your catchment or study area.  
- `Sites_Shapefile_Path` — points/polygons of monitoring sites (optional but recommended).

Below we:
- Try to **auto‑discover** shapefiles under `tests/Input data/` for a quick demo.
- Fall back to **editable placeholders** if none are found.


In [ ]:

from pathlib import Path
import glob, os

# --- Configuration flags ---
RUN_HEAVY_STEPS = False  # Set True to run online/download-heavy tasks (AWAP/AWRAL/Projections)

# --- Workspace (safe to create anywhere) ---
# Edit this to your own output directory (absolute path recommended)
CHM_Work_Space = str(Path.cwd() / "Output" / "Example_Run")

# --- Try to auto-discover test inputs ---
tests_input_dir = Path.cwd() / "tests" / "Input data"
catchment_candidates = []
site_candidates = []

if tests_input_dir.exists():
    # Common patterns; adjust if your filenames differ
    catchment_candidates = glob.glob(str(tests_input_dir / "**" / "*catchment*.shp"), recursive=True)
    site_candidates = glob.glob(str(tests_input_dir / "**" / "*site*.shp"), recursive=True)

# --- Fallback placeholders if auto-discovery fails ---
Catchment_Shapefile_Path = (catchment_candidates[0] if catchment_candidates 
                            else str(tests_input_dir / "Catchment.shp"))
Sites_Shapefile_Path = (site_candidates[0] if site_candidates 
                        else str(tests_input_dir / "Sites.shp"))

# Optional additional inputs for certain modules (edit as needed)
K_Factor_Path = str(tests_input_dir / "k_factor_g94.tif")   # Example path
P_Factor_Path = str(tests_input_dir / "P_factor_g94.tif")   # Example path
Monitoring_Data = str(tests_input_dir / "Monitoring_Data.xlsx")  # Example monitoring data (if present)

# Print resolved paths for verification
print("Workspace:", CHM_Work_Space)
print("Catchment:", Catchment_Shapefile_Path)
print("Sites    :", Sites_Shapefile_Path)
print("K factor :", K_Factor_Path)
print("P factor :", P_Factor_Path)
print("Monitoring data:", Monitoring_Data)
os.makedirs(CHM_Work_Space, exist_ok=True)



## 3) DEM & Terrain

This step generates DEM‑derived terrain metrics for the catchment and (optionally) extracts site‑level summaries.

**Inputs**: Catchment boundary, sites (optional), DEM source configured in the function.  
**Outputs**: Slope, aspect, ruggedness, topographic indices, and per‑site summaries (GeoTIFFs/CSV/GPKG, depending on implementation).


In [ ]:

# --- DEM and terrain analysis ---
try:
    dem_and_terrain(
        CHM_Work_Space,
        Catchment_Shapefile_Path,
        Sites_Shapefile_Path
    )
    print("DEM & terrain step: OK")
except Exception as e:
    print("DEM & terrain step: FAILED ->", e)



## 4) AWAP Historical (optional / online)

Downloads and processes AWAP historical precipitation and temperature for the catchment and, optionally, sites.

- **Toggle** with `RUN_HEAVY_STEPS`.  
- **Inputs**: Catchment, date range.  
- **Outputs**: NetCDF/CSV summaries per day (implementation‑specific).


In [ ]:

Start_Year = 2010
End_Year   = 2012

if RUN_HEAVY_STEPS:
    try:
        awap_historical_data(
            CHM_Work_Space,
            Catchment_Shapefile_Path,
            Sites_Shapefile_Path
        )
        print("AWAP historical: OK")
    except Exception as e:
        print("AWAP historical: FAILED ->", e)
else:
    print("AWAP historical: skipped (RUN_HEAVY_STEPS=False)")



## 5) Vegetation NDVI & C‑factor

Build vegetation indices (e.g., NDVI) and derive a C‑factor layer for erosion modeling.

**Inputs**: Catchment, sites (optional), time window and filters (e.g., cloud cover).  
**Outputs**: Index rasters, C‑factor rasters and site‑level summaries.


In [ ]:

# Example filters (edit to your study period and cloud threshold)
filter_query = "eo:cloud_cover < 20"
Datetime = "2024-01-01/2025-01-08"

try:
    veg_indices_and_c_factor(
        CHM_Work_Space,
        Catchment_Shapefile_Path,
        Sites_Shapefile_Path
    )
    print("Vegetation & C-factor: OK")
except Exception as e:
    print("Vegetation & C-factor: FAILED ->", e)



## 6) Surface & Groundwater Connectivity

Computes flow connectivity metrics (e.g., SDR upper bounds, IC parameters) and exports connectivity rasters and summaries.

**Tunable parameters** (set inside the function or exposed as arguments in your implementation).


In [ ]:

# Example parameters (if your function signature exposes them)
SDR_max = 0.8  # Maximum SDR (cap)
IC0 = 0.5      # Calibration parameter
k = 1          # Calibration parameter

try:
    surface_ground_water_connectivity(
        CHM_Work_Space,
        Catchment_Shapefile_Path,
        Sites_Shapefile_Path
    )
    print("Connectivity: OK")
except Exception as e:
    print("Connectivity: FAILED ->", e)



## 7) RUSLE & SDR‑RUSLE

Runs erosion modeling using RUSLE and propagates through sediment delivery (SDR).

**Inputs**: Catchment, sites, C‑factor, K‑factor, rainfall erosivity (R), slope/LS, and optional P‑factor.  
**Outputs**: Annual/periodic erosion rasters and statistics.


In [ ]:

Empirical_coefficient = 0.082  # Example empirical coefficient for erosivity

try:
    rusle_and_sdr_rusle(
        CHM_Work_Space,
        Catchment_Shapefile_Path,
        Sites_Shapefile_Path,
        K_Factor_Path,
        P_Factor_Path
    )
    print("RUSLE & SDR-RUSLE: OK")
except Exception as e:
    print("RUSLE & SDR-RUSLE: FAILED ->", e)



## 8) AWRAL Historical & Projections (optional / online)

Processes AWRAL historical variables (e.g., ET, runoff, soil moisture) and the AWRAL projections.

These steps can be data‑intensive and may require authentication or long downloads; keep them **off** for a quick smoke test.


In [ ]:

Start_Year = 2010
End_Year   = 2011

if RUN_HEAVY_STEPS:
    try:
        awra_historical_data(
            CHM_Work_Space,
            Catchment_Shapefile_Path,
            Start_Year,
            End_Year
        )
        print("AWRAL historical: OK")
    except Exception as e:
        print("AWRAL historical: FAILED ->", e)

    try:
        awra_projections_data(
            CHM_Work_Space,
            Catchment_Shapefile_Path
        )
        print("AWRAL projections: OK")
    except Exception as e:
        print("AWRAL projections: FAILED ->", e)
else:
    print("AWRAL historical/projections: skipped (RUN_HEAVY_STEPS=False)")



## 9) Risk Profiles — Bushfire, Roads, Landuse

Generate cumulative risk profiles (e.g., combining SDR/TWI/NDVI with hazard layers).

**Inputs**: Catchment (and possibly annual layers or scenario rasters).  
**Outputs**: Per‑class/channel/site risk summaries and plots.


In [ ]:

# Historical bushfire
try:
    historical_bushfire_risk_profile(
        CHM_Work_Space,
        Catchment_Shapefile_Path
    )
    print("Bushfire risk profile: OK")
except Exception as e:
    print("Bushfire risk profile: FAILED ->", e)

# National roads
try:
    national_roads_risk_profile(
        CHM_Work_Space,
        Catchment_Shapefile_Path
    )
    print("National roads risk profile: OK")
except Exception as e:
    print("National roads risk profile: FAILED ->", e)

# Landuse
try:
    landuse_risk_profile(
        CHM_Work_Space,
        Catchment_Shapefile_Path
    )
    print("Landuse risk profile: OK")
except Exception as e:
    print("Landuse risk profile: FAILED ->", e)



## 10) Append Monitoring Data to Site Layers

This utility attaches monitoring tables (e.g., Excel/CSV) to site geometries to maintain a single geospatial record per site.

**Inputs**: `Monitoring_Data` (tabular), sites layer (GeoPackage/Shapefile).  
**Outputs**: Updated site datasets under your workspace.


In [ ]:

try:
    appending_monitoring_data(
        CHM_Work_Space,
        Catchment_Shapefile_Path,
        Monitoring_Data
    )
    print("Append monitoring data: OK")
except Exception as e:
    print("Append monitoring data: FAILED ->", e)



## 11) What next?

- Inspect outputs under your workspace: `Output/Example_Run/<Catchment Name>/...`  
- Re‑run with `RUN_HEAVY_STEPS = True` for the full data download pipeline.  
- Replace example inputs with your real project paths.

If you hit import or path errors, double‑check that the package is installed in *this* kernel and that your input paths exist.
